In [1]:
import finnhub
import os
from dotenv import load_dotenv

load_dotenv()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

data = finnhub_client.company_basic_financials(symbol="AAPL", metric="all")

In [2]:
# See what's inside annual vs quarterly
print("=== SERIES KEYS ===")
for k in data["series"].keys():
    print(f"  {k}")

print("\n=== ANNUAL KEYS ===")
for k in data["series"]["annual"].keys():
    print(f"  {k}")

print("\n=== QUARTERLY KEYS ===")
for k in data["series"]["quarterly"].keys():
    print(f"  {k}")

=== SERIES KEYS ===
  annual
  quarterly

=== ANNUAL KEYS ===
  bookValue
  cashRatio
  currentRatio
  ebitPerShare
  eps
  ev
  evEbitda
  evRevenue
  fcfMargin
  grossMargin
  inventoryTurnover
  longtermDebtTotalAsset
  longtermDebtTotalCapital
  longtermDebtTotalEquity
  netDebtToTotalCapital
  netDebtToTotalEquity
  netMargin
  operatingMargin
  payoutRatio
  pb
  pe
  pfcf
  pretaxMargin
  ps
  ptbv
  quickRatio
  receivablesTurnover
  roa
  roe
  roic
  rotc
  salesPerShare
  sgaToSale
  tangibleBookValue
  totalDebtToEquity
  totalDebtToTotalAsset
  totalDebtToTotalCapital
  totalRatio

=== QUARTERLY KEYS ===
  assetTurnoverTTM
  bookValue
  cashRatio
  currentRatio
  ebitPerShare
  eps
  ev
  evEbitdaTTM
  evRevenueTTM
  fcfMargin
  fcfPerShareTTM
  grossMargin
  inventoryTurnoverTTM
  longtermDebtTotalAsset
  longtermDebtTotalCapital
  longtermDebtTotalEquity
  netDebtToTotalCapital
  netDebtToTotalEquity
  netMargin
  operatingMargin
  payoutRatioTTM
  pb
  peTTM
  pfcfTTM
 

In [3]:
# Pick one metric from each to see the time-series shape
first_annual_key = next(iter(data["series"]["annual"]))
first_quarterly_key = next(iter(data["series"]["quarterly"]))

print(f"=== ANNUAL SAMPLE: {first_annual_key} ===")
for entry in data["series"]["annual"][first_annual_key][:3]:
    print(f"  {entry}")

print(f"\n=== QUARTERLY SAMPLE: {first_quarterly_key} ===")
for entry in data["series"]["quarterly"][first_quarterly_key][:3]:
    print(f"  {entry}")

=== ANNUAL SAMPLE: bookValue ===
  {'period': '2025-09-27', 'v': 73733}
  {'period': '2024-09-28', 'v': 56950}
  {'period': '2023-09-30', 'v': 62146}

=== QUARTERLY SAMPLE: assetTurnoverTTM ===
  {'period': '2025-12-27', 'v': 1.2435}
  {'period': '2025-09-27', 'v': 1.2186}
  {'period': '2025-06-28', 'v': 1.1915}


In [4]:
print(f"Total flat metrics:       {len(data['metric'])}")
print(f"Annual series metrics:    {len(data['series']['annual'])}")
print(f"Quarterly series metrics: {len(data['series']['quarterly'])}")

Total flat metrics:       132
Annual series metrics:    38
Quarterly series metrics: 40


### ─────────────────────────────────────────────
### SECTION 2 — PRODUCTION RUN (ALL 60 COMPANIES)
### ─────────────────────────────────────────────

In [5]:
import finnhub
import os
import time
from collections import defaultdict
from dotenv import load_dotenv
load_dotenv()

finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

tickers = [
    # DEFENSE - High Lobby
    "LMT", "RTX", "NOC", "GD", "BA", "LHX", "LDOS", "HII", "BAESY", "SAIC",
    # DEFENSE - Low Lobby
    "TXT", "TDG", "HEI", "DRS", "KTOS", "AVAV", "MRCY", "CW", "MOG.A", "DCO",
    # ENERGY - High Lobby
    "XOM", "CVX", "COP", "OXY", "BP", "NEE", "D", "DUK", "HAL", "BKR",
    # ENERGY - Low Lobby
    "SLB", "VLO", "PSX", "EOG", "FANG", "DVN", "CTRA", "AR", "CHRD", "MTDR",
    # TECH - High Lobby
    "MSFT", "AMZN", "GOOGL", "IBM", "ORCL", "PLTR", "BAH", "CACI", "PSN", "CRM",
    # TECH - Low Lobby
    "AAPL", "META", "NVDA", "CSCO", "PANW", "CRWD", "SNOW", "DDOG", "NET", "TWLO"
]

duplicates = [t for t in tickers if tickers.count(t) > 1]
assert not duplicates, f"Duplicate tickers found: {duplicates}"

results = {}
errors  = []

# ── metric block tracking ──────────────────────────────────────────────────
metric_present = defaultdict(int)
metric_null    = defaultdict(int)
metric_types   = defaultdict(set)

# ── series structure tracking ──────────────────────────────────────────────
series_present          = 0   # tickers where series key exists
annual_present          = 0   # tickers where series.annual exists and is non-empty
quarterly_present       = 0   # tickers where series.quarterly exists and is non-empty
annual_entry_types      = defaultdict(set)   # field → set of value types inside annual entries
quarterly_entry_types   = defaultdict(set)   # field → set of value types inside quarterly entries
annual_null             = defaultdict(int)
quarterly_null          = defaultdict(int)

total = len(tickers)

for i, ticker in enumerate(tickers):
    try:
        data = finnhub_client.company_basic_financials(symbol=ticker, metric="all")

        if not data or not data.get("metric"):
            errors.append((ticker, "empty response — metric block missing"))
            results[ticker] = {}
        else:
            results[ticker] = data

            # ── metric block ──────────────────────────────────────────────
            for field, value in data["metric"].items():
                metric_present[field] += 1
                if value is None or value == "":
                    metric_null[field] += 1
                else:
                    metric_types[field].add(type(value).__name__)

            # ── series block ──────────────────────────────────────────────
            series = data.get("series", {})
            if series:
                series_present += 1

                annual = series.get("annual", {})
                if annual:
                    annual_present += 1
                    # Sample first entry of each annual metric to check types
                    for metric_name, entries in annual.items():
                        if entries:
                            sample = entries[0]
                            for k, v in sample.items():
                                if v is None:
                                    annual_null[k] += 1
                                else:
                                    annual_entry_types[k].add(type(v).__name__)

                quarterly = series.get("quarterly", {})
                if quarterly:
                    quarterly_present += 1
                    for metric_name, entries in quarterly.items():
                        if entries:
                            sample = entries[0]
                            for k, v in sample.items():
                                if v is None:
                                    quarterly_null[k] += 1
                                else:
                                    quarterly_entry_types[k].add(type(v).__name__)

    except Exception as e:
        errors.append((ticker, f"API error: {str(e)}"))
        results[ticker] = {}

    if i < len(tickers) - 1:
        time.sleep(1)

# ─────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────
successful    = len([r for r in results.values() if r])
empty_tickers = [t for t, r in results.items() if not r]

print(f"✅ Successfully pulled: {successful} / {total}")
print(f"❌ Errors:              {len(errors)}")
print(f"📭 Empty responses:     {empty_tickers if empty_tickers else 'None'}\n")

if errors:
    print("─── Errors ───")
    for ticker, msg in errors:
        print(f"   {ticker}: {msg}")
    print()

# ─────────────────────────────────────────
# SECTION 1 — TOP-LEVEL STRUCTURE
# ─────────────────────────────────────────
print("─── Top-Level Structure ───")
print(f"  metric block present:      {successful}/{total} tickers")
print(f"  series block present:      {series_present}/{total} tickers")
print(f"  series.annual present:     {annual_present}/{total} tickers")
print(f"  series.quarterly present:  {quarterly_present}/{total} tickers\n")

# ─────────────────────────────────────────
# SECTION 2 — METRIC BLOCK DECISION TABLE
# ─────────────────────────────────────────
print("─── Metric Block — Pydantic Decision Table ───")
print(f"{'Field':<45} {'Present':>10} {'Null':>8} {'Types':<20} Recommendation")
print("─" * 110)

for field in sorted(metric_present.keys()):
    present    = metric_present[field]
    null_count = metric_null.get(field, 0)
    types      = ", ".join(metric_types.get(field, {"unknown"}))
    always_present = present == successful
    ever_null      = null_count > 0

    if not always_present:
        rec = "Optional  ← missing from some tickers"
    elif ever_null:
        rec = "Optional  ← null in some tickers"
    else:
        rec = "Required  ← always present and never null"

    print(f"{field:<45} {present:>7}/{total}  {null_count:>5} null   {types:<20} {rec}")

# ─────────────────────────────────────────
# SECTION 3 — SERIES ENTRY SHAPE
# ─────────────────────────────────────────
print("\n─── Series Entry Shape (annual) ───")
print(f"{'Field':<15} {'Null count':>12} {'Types':<20} Recommendation")
print("─" * 65)
all_entry_keys = set(annual_entry_types.keys()) | set(annual_null.keys())
for field in sorted(all_entry_keys):
    null_count = annual_null.get(field, 0)
    types      = ", ".join(annual_entry_types.get(field, {"unknown"}))
    rec        = "Optional" if null_count > 0 else "Required"
    print(f"{field:<15} {null_count:>10} null   {types:<20} {rec}")

print("\n─── Series Entry Shape (quarterly) ───")
print(f"{'Field':<15} {'Null count':>12} {'Types':<20} Recommendation")
print("─" * 65)
all_quarterly_keys = set(quarterly_entry_types.keys()) | set(quarterly_null.keys())
for field in sorted(all_quarterly_keys):
    null_count = quarterly_null.get(field, 0)
    types      = ", ".join(quarterly_entry_types.get(field, {"unknown"}))
    rec        = "Optional" if null_count > 0 else "Required"
    print(f"{field:<15} {null_count:>10} null   {types:<20} {rec}")

✅ Successfully pulled: 60 / 60
❌ Errors:              0
📭 Empty responses:     None

─── Top-Level Structure ───
  metric block present:      60/60 tickers
  series block present:      60/60 tickers
  series.annual present:     60/60 tickers
  series.quarterly present:  60/60 tickers

─── Metric Block — Pydantic Decision Table ───
Field                                            Present     Null Types                Recommendation
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
10DayAverageTradingVolume                          60/60      0 null   float                Required  ← always present and never null
13WeekPriceReturnDaily                             60/60      0 null   float                Required  ← always present and never null
26WeekPriceReturnDaily                             60/60      0 null   float                Required  ← always present and never null
3MonthADReturnStd                                  6